# LLaMA-style modern decoder (RoPE + RMSNorm + SwiGLU + GQA)

**Paper:** LLaMA: Open and Efficient Foundation Language Models
**Difficulty:** hard · **Estimated time:** 14 hours

## Why this exercise
Once you have GPT-2 working, LLaMA shows what every modern open LLM actually uses: rotary positions, RMSNorm pre-normalization, SwiGLU MLPs, and grouped-query attention.

## Learning objective
Implement the modern LLaMA decoder stack from scratch and train it end-to-end on synthetic data.

## Context
LLaMA is the recipe shared by Mistral, Qwen, DeepSeek, and most other open models. Every change vs GPT-2 has a measurable reason; building it from scratch lets you feel them.

## Prerequisites
- RoPE rotation of Q/K
- RMSNorm vs LayerNorm
- SwiGLU activation
- Grouped-query attention

## Concept ladder
1. RMSNorm normalizes by RMS, no mean subtraction
2. RotaryEmbedding produces cos/sin tables; apply_rotary rotates Q/K pairs
3. Grouped-query attention shares K/V across multiple Q heads
4. SwiGLU = (silu(W1 x) * W2 x) W3
5. Stack LlamaBlocks and finish with an LM head

## Tasks
- Task 1: RMSNorm and RotaryEmbedding (cos/sin tables of shape (seq_len, head_dim)).
- Task 2: LlamaAttention with GQA and RoPE applied to Q and K.
- Task 3: SwiGLU MLP and the full LlamaBlock (pre-norm RMSNorm + attention + RMSNorm + SwiGLU + residuals).
- Task 4: LlamaModel stacking blocks; pipeline returns train_loss and val_loss.

## Milestones
1. **Forward shape**
   - Goal: LlamaModel returns (B, T, V) logits.
   - Check: `test_llama_forward_shape passes.`
2. **Trains on one batch**
   - Goal: Loss halves on a fixed batch within 80 AdamW steps.
   - Check: `test_llama_overfits_one_batch passes.`
3. **Pipeline runs**
   - Goal: run_pipeline returns finite train_loss and val_loss.
   - Check: `test_pipeline_runs_and_reports_metrics passes.`

## Setup
CPU is enough at the tiny config. Synthetic vocab + sequence data.

## How to work
- Implement the TODOs in the code cell below.
- Run the cell; the included tests identify the first failing requirement.
- Keep the implementation in one file-equivalent cell unless the exercise says otherwise.
- Finish only when the end-to-end pipeline and all checks pass.

In [ ]:
# Colab setup — PyTorch is normally preinstalled.
try:
    import torch
except ImportError:
    %pip install -q torch

import torch
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Project: LLaMA-style decoder LM from scratch (RoPE + RMSNorm + SwiGLU + GQA)
# Paper: LLaMA: Open and Efficient Foundation Language Models (Touvron et al., 2023)
# Goal: implement the full modern LLaMA architecture AND the end-to-end pretraining pipeline
# Compute: CPU is enough at the tiny config used by the tests
# Rule: keep this as ONE file.
# Run when done: python assignment.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# =========================
# STUDENT CODE STARTS HERE
# =========================


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        raise AssertionError("TODO: implement RMSNorm.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement RMSNorm.forward")


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        raise AssertionError("TODO: implement RotaryEmbedding.__init__")

    def forward(self, seq_len):
        raise AssertionError("TODO: implement RotaryEmbedding.forward")


class LlamaAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, num_kv_heads, max_seq_len):
        super().__init__()
        raise AssertionError("TODO: implement LlamaAttention.__init__")

    def forward(self, x, rope):
        raise AssertionError("TODO: implement LlamaAttention.forward")


class SwiGLU(nn.Module):
    def __init__(self, embed_dim, mlp_dim):
        super().__init__()
        raise AssertionError("TODO: implement SwiGLU.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement SwiGLU.forward")


class LlamaBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, num_kv_heads, mlp_dim, max_seq_len):
        super().__init__()
        raise AssertionError("TODO: implement LlamaBlock.__init__")

    def forward(self, x, rope):
        raise AssertionError("TODO: implement LlamaBlock.forward")


class LlamaModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len, embed_dim, depth, num_heads, num_kv_heads, mlp_dim):
        super().__init__()
        raise AssertionError("TODO: implement LlamaModel.__init__")

    def forward(self, idx):
        raise AssertionError("TODO: implement LlamaModel.forward")


def get_dataloaders(num_train, num_val, vocab_size, seq_len, batch_size):
    raise AssertionError("TODO: implement get_dataloaders")


def train_one_epoch(model, loader, optimizer, device):
    raise AssertionError("TODO: implement train_one_epoch")


def evaluate(model, loader, device):
    raise AssertionError("TODO: implement evaluate")


def run_pipeline():
    raise AssertionError("TODO: implement run_pipeline that wires data, model, optimizer, training and evaluation, and returns a dict with train_loss and val_loss")


# =========================
# STUDENT CODE ENDS HERE
# TESTS START BELOW
# =========================


TINY_CONFIG = dict(
    vocab_size=64,
    max_seq_len=16,
    embed_dim=64,
    depth=2,
    num_heads=4,
    num_kv_heads=2,
    mlp_dim=128,
)


def test_llama_forward_shape():
    model = LlamaModel(**TINY_CONFIG)
    idx = torch.randint(0, TINY_CONFIG["vocab_size"], (2, TINY_CONFIG["max_seq_len"]))
    logits = model(idx)
    expected = (2, TINY_CONFIG["max_seq_len"], TINY_CONFIG["vocab_size"])
    assert logits.shape == expected, f"logits shape was {logits.shape}, expected {expected}"
    assert torch.isfinite(logits).all(), "logits contain NaN/inf"


def test_llama_overfits_one_batch():
    torch.manual_seed(0)
    model = LlamaModel(**TINY_CONFIG)
    idx = torch.randint(0, TINY_CONFIG["vocab_size"], (4, TINY_CONFIG["max_seq_len"]))
    targets = torch.randint(0, TINY_CONFIG["vocab_size"], (4, TINY_CONFIG["max_seq_len"]))
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    V = TINY_CONFIG["vocab_size"]
    initial = F.cross_entropy(model(idx).reshape(-1, V), targets.reshape(-1)).item()
    for _ in range(80):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(idx).reshape(-1, V), targets.reshape(-1))
        loss.backward()
        optimizer.step()
    final = F.cross_entropy(model(idx).reshape(-1, V), targets.reshape(-1)).item()
    assert final < 0.5 * initial, f"final loss {final:.4f} should be much lower than initial {initial:.4f}"


def test_pipeline_runs_and_reports_metrics():
    torch.manual_seed(0)
    result = run_pipeline()
    assert isinstance(result, dict), "run_pipeline must return a dict"
    assert "train_loss" in result and "val_loss" in result, "result must contain train_loss and val_loss"
    val_loss = float(result["val_loss"])
    assert val_loss == val_loss, "val_loss is NaN"


def run_all_tests():
    print("Running LLaMA pipeline checks...")
    test_llama_forward_shape()
    print("1/3 LLaMA forward shape passed")
    test_llama_overfits_one_batch()
    print("2/3 model can overfit a single batch")
    test_pipeline_runs_and_reports_metrics()
    print("3/3 end-to-end pipeline returns metrics")
    print("All checks passed. You can mark this assignment done.")


if __name__ == "__main__":
    run_all_tests()
